In [1]:
!pip install pypulseq

^C


In [3]:
import numpy as np 
import pypulseq as pq 
import matplotlib.pyplot as plt

In [10]:
import numpy as np 
import pypulseq as pq 
import matplotlib.pyplot as plt 

# maximum gradient amplitude [mT/m]
max_grad = 40.0

# maximum gradient slew rate [T/m/s]
# set super low here to eliminate PNS risks 
max_slew = 90.0

# duration of each sample in RF pulse [s]
rf_raster_time = 2.0e-6 

# necessary post-rf delay [s]
rf_ringdown_time = 60.0e-6

# necessary pre-rf delay [s]
rf_dead_time = 100.0e-6 

# necessary pre-adc delay [s] 
adc_dead_time = 40.0e-6 

# adc raster time [s]
adc_raster_time = 2.0e-6 

# gradient raster time [s]
grad_raster_time = 4.0e-6 

# total duration of each block must be evenly divisible by this number
block_duration_raster = 4.0e-6 

# delay inserted between segments for GE systems 
# added by GE pge2 interpreter 
end_of_segment_delay = 116.0e-6

# make the system object for PyPulseq 
system = pq.Opts(max_grad=max_grad,
                     grad_unit='mT/m',
                     max_slew=max_slew,
                     slew_unit='T/m/s',
                     rf_ringdown_time=rf_ringdown_time,
                     rf_dead_time=rf_dead_time,
                     rf_raster_time=rf_raster_time,
                     adc_dead_time=adc_dead_time,
                     adc_raster_time=adc_raster_time,
                     grad_raster_time=grad_raster_time,
                     block_duration_raster=block_duration_raster)

fovx = 0.22 # frequency encoding FOV [m]
fovy = 0.22 # phase encoding FOV [m]
dz = 0.005  # slice thickness [m]

# imaging matrix sizes 
nx = 128 
ny = 128 

# TODO: choose inversion times 
# T1 = [.1-2]s
TI = np.linspace(0.1, 2, 6, dtype=np.float32)

# TODO: choose this
TR = 5.0 # repetition time [s]

# TODO: choose this
flip_angle = 30.0 
alpha = flip_angle * np.pi / 180 

# TODO: choose this (make a multiple of 2 microseconds)
adc_dwell_time = 6.0e-6

# TODO: calculate frequency encoding gradient amplitude in units of Hz/m 
kx = 1/fovx
Gx_Hz_m = kx / adc_dwell_time 

# TODO: calculate duration of flat part of frequency encoding gradient 
Gx_flat_time = nx * adc_dwell_time

# make the frequency encoding gradient 
Gx = pq.make_trapezoid(channel='x', amplitude=Gx_Hz_m, flat_time=Gx_flat_time, system=system)

# Data collection object
adc = pq.make_adc(num_samples=nx, delay=Gx.rise_time, duration=Gx_flat_time, system=system)

# TODO: make the frequency encoding prephaser 
# Gx.area
area_Gx_pre = -0.5 * Gx.area # calculate this! 
Gx_pre = pq.make_trapezoid(channel='x', area=area_Gx_pre, system=system)

# TODO: calculate phase encoding areas (units of 1/m)
delta_ky = 1/fovy
phase_areas = np.arange(-ny/2, ny/2)*delta_ky # [m^-1] you make this, it should be a numpy array with length of ny
max_phase_area = np.max(np.abs(phase_areas))
phase_scales = phase_areas / max_phase_area # factor by which to scale amplitude of Gy gradient for each phase encoding step

# make the phase encoding gradient 
Gy = pq.make_trapezoid(channel='y', area=max_phase_area, system=system)

# TODO: make the inversion pulse 
inv_duration = 1.0e-3 
inv_flip_angle = np.pi # TODO: set this
rf_inv = pq.make_block_pulse(flip_angle=inv_flip_angle, duration=inv_duration, system=system)

# TODO: make the slice-selective excitation pulse 
exc_bw = 2000 # TODO: choose this
exc_duration = 0.003 # TODO: choose this 
exc_tbw = exc_bw * exc_duration 
rf_exc, gz, _ = pq.make_sinc_pulse(flip_angle=alpha, duration=exc_duration, time_bw_product=exc_tbw, slice_thickness=dz, return_gz=True, system=system)

area_Gz_pre = -0.5 * gz.area
Gz_pre = pq.make_trapezoid(channel='z', area=area_Gz_pre, system=system)

area_Gz_spoil = 4/dz
Gz_spoil = pq.make_trapezoid(channel='z', area=area_Gz_spoil, system=system)

delay_TI = TI - 0.5*pq.calc_duration(rf_inv) - 0.5*pq.calc_duration(gz)
delay_TI = (np.round(delay_TI/grad_raster_time)).astype(np.int32)*grad_raster_time

TI_delay_object = []
for t in range(len(TI)):
    TI_delay_object.append(pq.make_delay(delay_TI[t]))

C:\Users\jmarasco\AppData\Local\Temp\ipykernel_13276\1226771445.py:102: UserWarning: Specified RF delay 0.00 us is less than the dead time 100 us. Delay was increased to the dead time.
  rf_inv = pq.make_block_pulse(flip_angle=inv_flip_angle, duration=inv_duration, system=system)
C:\Users\jmarasco\AppData\Local\Temp\ipykernel_13276\1226771445.py:108: UserWarning: Specified RF delay 0.00 us is less than the dead time 100 us. Delay was increased to the dead time.
  rf_exc, gz, _ = pq.make_sinc_pulse(flip_angle=alpha, duration=exc_duration, time_bw_product=exc_tbw, slice_thickness=dz, return_gz=True, system=system)


In [11]:
seq = pq.Sequence(system=system)

for inv in range(TI.size):
    for y in range(ny):
        seq.add_block(rf_inv)
        
        seq.add_block(TI_delay_object[inv])
    
        seq.add_block(rf_exc, gz)
    
        seq.add_block(Gz_pre, Gx_pre, pq.scale_grad(Gy, scale=phase_scales[y]))

        seq.add_block(Gx, adc)

        seq.add_block(Gz_spoil, pq.scale_grad(Gy, scale=-phase_scales[y]))

In [8]:
%matplotlib tk

In [12]:
seq.plot()